# Lesson 02 - Esplorare il Microsoft Agent Framework

Il **Microsoft Agent Framework (MAF)** è un framework unificato per costruire agenti AI. Fornisce un'architettura pulita e componibile con quattro componenti fondamentali:

- **Client** – si connette a un endpoint di modello AI e gestisce la comunicazione
- **Agent** – incapsula un client con istruzioni e definizioni degli strumenti
- **Tools** – estendono le capacità dell'agente con funzioni personalizzate che il modello può chiamare
- **Session** – mantiene la cronologia della conversazione per interazioni multi-turno

In questa lezione, costruiremo un **agente per prenotazioni di viaggio** che verifica la disponibilità della destinazione utilizzando questi concetti.


## Configurazione


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Comprendere l'Architettura del Framework Agent

Il Microsoft Agent Framework segue un'architettura stratificata:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Un `AzureAIProjectAgentProvider` si collega a un deployment Azure OpenAI. Gestisce l'autenticazione, la formattazione delle richieste e l'analisi delle risposte.
2. **Agent** – Creato dal client tramite `provider.create_agent()`, l'agent combina l'accesso al modello con istruzioni (prompt di sistema) e strumenti.
3. **Tools** – Funzioni Python decorate con `@tool` che l'agent può invocare per eseguire azioni o recuperare dati.
4. **Session** – Un oggetto `AgentSession` (creato tramite `agent.create_session()`) che memorizza la cronologia della conversazione, consentendo un dialogo multi-turno in cui l'agent ricorda il contesto precedente.

Costruiamo ogni livello passo dopo passo.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Aggiunta di Strumenti con il Decoratore @tool

Gli strumenti consentono agli agenti di eseguire azioni oltre alla generazione di testo. Il decoratore `@tool` trasforma una normale funzione Python in qualcosa che l'agente può chiamare.

Punti chiave:
- Usa `Annotated[type, "description"]` in modo che il modello comprenda ogni parametro.
- La docstring diventa la descrizione dello strumento che il modello vede.
- `approval_mode="never_require"` significa che lo strumento viene eseguito automaticamente senza conferma dell'utente.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creazione di un agente con gli strumenti

Ora combiniamo il client, le istruzioni e gli strumenti in un agente. Le `instructions` fungono da prompt di sistema — definiscono la persona e il comportamento dell’agente.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversazioni multi-turno con sessioni

Un `AgentSession` (creato tramite `agent.create_session()`) tiene traccia di tutti i messaggi in una conversazione. Passando la stessa sessione a ogni chiamata di `agent.run()`, l'agente ha accesso all'intera cronologia della conversazione e può fare riferimento ai messaggi precedenti.

Passiamo `tools=[check_destination_availability]` in modo che l'agente possa chiamare il nostro verificatore di disponibilità durante ogni turno.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

In questa lezione hai esplorato i quattro pilastri del Microsoft Agent Framework:

| Concetto | Cosa Hai Imparato |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` si connette ad Azure OpenAI con autenticazione basata su credenziali |
| **Agent** | `provider.create_agent()` combina una connessione al modello con istruzioni e un nome |
| **Tools** | Il decoratore `@tool` espone funzioni Python che l'agente può chiamare |
| **Session** | `agent.create_session()` mantiene la cronologia della conversazione attraverso più turni |

Questi blocchi fondamentali si combinano per creare agenti in grado di sostenere conversazioni naturali, chiamare funzioni esterne e mantenere il contesto — la base per modelli agentici più avanzati nelle lezioni successive.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avvertenza**:  
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci per garantire l’accuratezza, si prega di tenere presente che le traduzioni automatiche possono contenere errori o inesattezze. Il documento originale nella sua lingua madre deve essere considerato la fonte autorevole. Per informazioni critiche si raccomanda la traduzione professionale effettuata da un essere umano. Non ci assumiamo alcuna responsabilità per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
